In [ ]:
"""
~ Download and clean the CFT Version 2.0
~ Clean erroneous shapes
~ Set geometry snap tolerance
~ Clean geometries, snap

author: maxwell.cook@colostate.edu
"""

import os, sys
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import numpy as np
import shapely

from os.path import join
from pathlib import Path

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from functions import *  # imports all custom functions

# use the current working directory
boxdir = Path.cwd().parents[3] # primary base box directory
projdir = Path.cwd().parents[0] # project directory
print(f"Box directory set to: {boxdir}")
print(f"Project directory set to: {projdir}")

proj_crs = 26913  # NAD83 UTM Zone 13N

m2_to_acres = 0.000247105 # conversion (multiplication) of m2 to acres
min_acres = (30*30)*m2_to_acres # approximate size of one 30m pixel
print(f"Minimum acre threshold for valid treatment: {min_acres}")

In [ ]:
# --- Download the CFT from the ArcGIS REST service (or load)
from tealom.utils import get_feature_service_gdf

out_fp = join(projdir,'data/spatial/raw/CFTv2_CO_AllTreatments.gpkg')

if not os.path.exists(out_fp):
    # All treatment data
    cft_url = 'https://services3.arcgis.com/unh8qj8OkZd7wlfi/arcgis/rest/services/Forest_Tracker_v2/FeatureServer'
    cft = get_feature_service_gdf(url=cft_url, layer=10) # layer=10 based on URL parameters
    cft = cft.to_crs(epsg=proj_crs)
    print(f"Extracted {len(cft)} treatment features from the REST service.")

    # --- Save the file locally
    os.makedirs(os.path.dirname(out_fp), exist_ok=True)
    cft.to_file(out_fp)
    print(f"Saved to {out_fp}")

else:
    # load the downloaded file
    cft = gpd.read_file(out_fp).to_crs(proj_crs)
    print(f"CFT already downloaded to {out_fp}"
          f"\n\tNumber of treatment polygons: {len(cft)}")

# --- Fix any invalid geometries
cft['geometry'] = cft.geometry.buffer(0)

# print some summaries
# --- Print some summaries of the data:
print(f"\nN = {len(cft)} ({cft['YEAR_COMP'].min()}-{cft['YEAR_COMP'].max()})\n")
print(f"~\t# Agencies: {len(cft['AGENCY_C'].unique())}")
print(f"~\t# Unique project names: {len(cft['PRJ_NAME'].unique())}")
print(f"~\t# Funding sources: {len(cft['FUND_SOURCE'].unique())}")
print(f"~\tTotal management acres: {round(cft['ACRES_MGT'].sum())}")
print(f"~\tTotal GIS acres: {round(cft['ACRES_GIS'].sum())}")
print(f"~\tManagement types: {cft['MGT_TYPE'].unique()}")
print(f"~\tActivities:\n\t\t{cft['ACTIVITY'].unique()}")
print(f"~\tCRS: {cft.crs}")
print(f"~\tCFT columns:\n{cft.columns}")

In [ ]:
"""
Heatmap of vegetation and fuels treatments in CO
"""

from matplotlib.colors import LogNorm
from scipy.stats import gaussian_kde

grid_size = 5000 # 5km grids

# --- Load CO counties
fp = join(boxdir,'MCC/data/boundaries/political/TIGER/'
                 'tl_2024_us_county/tl_2024_us_county.shp')
counties = gpd.read_file(fp).to_crs(proj_crs)
co_counties = counties[counties['STATEFP'] == '08']  # CO FIPS code

# --- Treatments representative point
cft_pt = cft.copy().to_crs(proj_crs)
cft_pt['geometry'] = cft_pt.geometry.buffer(0) # fixes geometries
cft_pt['geometry'] = cft.geometry.centroid
lon = cft_pt.geometry.x.values
lat = cft_pt.geometry.y.values

# --- Create a regular grid over Colorado
bounds = co_counties.total_bounds  # [minx, miny, maxx, maxy]
xx = np.arange(bounds[0], bounds[2], grid_size)
yy = np.arange(bounds[1], bounds[3], grid_size)
heatmap, x_edges, y_edges = np.histogram2d(cft_pt.geometry.x,
                                           cft_pt.geometry.y,
                                           bins=[xx, yy])

# --- Plot it!

# --- Mask zeros and apply log norm
heatmap_m = np.ma.masked_where(heatmap == 0, heatmap)
vmin = np.min(heatmap_m[heatmap_m > 0])
vmax = np.max(heatmap_m)

fig, ax = plt.subplots(figsize=(8,6))
vmax = np.max(heatmap)
img = ax.imshow(heatmap_m.T,
                extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                origin='lower', cmap='YlOrRd',
                norm=LogNorm(vmin=vmin, vmax=vmax),
                alpha=0.85, interpolation='bilinear')
co_counties.plot(ax=ax, color='none',
              edgecolor='black', linewidth=0.5)
ax.set_title('Colorado Treatment Heatmap (2000-2024)', fontsize=11)
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xlim(bounds[0], bounds[2])
ax.set_ylim(bounds[1], bounds[3])
ax.set_aspect('equal')
ax.grid(False)
cbar = plt.colorbar(img, ax=ax, label='Treatments per 5km² Cell (log scale)',
                    pad=0.03, shrink=0.75)

out_png = join(projdir,'figures/CO_Treatment_Heatmap.png')
os.makedirs(os.path.dirname(out_png), exist_ok=True)
plt.savefig(out_png)

plt.show()

## Filter non-treatment polygons

In the CFT, there are often cases where the GIS acres (polygon) vary dramitacally from the reported management acres. For downstream modeling, this is an issue. In the next section, we identify potential erroneous polygon records (e.g., large management units, NEPA boundaries, or private parcel boundaries) using a log-log regression GIS Acres ~ MGT Acres.

In [ ]:
# --- Explore the GIS / MGT acres mismatches
print(f"Proportion of NaN MGT_ACRES: {len(cft[cft['ACRES_MGT'].isna()])/len(cft)*100}")
print(f"Proportion of 0 MGT_ACRES: {round(len(cft[cft['ACRES_MGT']==0])/len(cft)*100,2)}")
print(f"\nProportion of NaN GIS_ACRES: {len(cft[cft['ACRES_GIS'].isna()])/len(cft)*100}")
print(f"Proportion of 0 GIS_ACRES: {round(len(cft[cft['ACRES_GIS']==0])/len(cft)*100,2)}")

In [ ]:
# --- Keep valid records (>min_acres/2 ACRES_GIS) -- Default to one half 30m pixel (0.22 acres)
cft['gis_acres'] = cft.geometry.area * m2_to_acres # re-calculate the GIS acres
cft_ = cft[cft['gis_acres'] > min_acres/2]
# --- Look at the acres distribution (GIS)
print(f"Distribution of GIS acres:\n{cft_['gis_acres'].describe()}")

In [ ]:
# --- Calculate the ratio between GIS / MGT acres, plot
cft_['acres_diff'] = cft_['ACRES_GIS'] - cft_['ACRES_MGT'] # difference in acres
cft_['log_ratio'] = np.log2(cft_['ACRES_GIS'] / cft_['ACRES_MGT'])
pd.set_option('display.float_format', lambda x: '{:,.3f}'.format(x))
print(f"\nSummary of GIS/MGT acre difference:\n{cft_['acres_diff'].describe()}\n")

# --- Log-log plot of GIS/MGT acres
plt.figure(figsize=(5,3))
plt.loglog(cft_['ACRES_GIS'], cft_['ACRES_MGT'], 'o', markersize=1.5)
plt.show()

In [ ]:
"""
Fit a log-log regression for GIS acres ~ MGT acres
~ use residuals to identify erroneous shapes
"""

from scipy import stats

# --- Filter to valid (non-zero, non-NaN) acre pairs
mask = (cft_['ACRES_GIS'] > 0) & (cft_['ACRES_MGT'] > 0) & \
       (cft_['ACRES_MGT'].notna()) & (cft_['ACRES_GIS'].notna())
df_valid = cft_[mask].copy()

# --- Get the log-scaled model variables
log_gis = np.log10(df_valid['ACRES_GIS'])
log_mgt = np.log10(df_valid['ACRES_MGT'])

# --- Fit OLS in log-log space
slope, intercept, r, p, se = stats.linregress(log_gis, log_mgt)
print(f"Log-log OLS: slope={slope:.3f}, intercept={intercept:.3f}, R²={r**2:.3f}")

# --- Get the fitted values and residuals
df_valid['log_gis'] = log_gis
df_valid['log_mgt'] = log_mgt
df_valid['log_ac_pred'] = intercept + slope * log_gis
df_valid['log_ac_resid'] = log_mgt - df_valid['log_ac_pred']

# --- Flag: residual-based (model-driven) + absolute floor
resid_std = df_valid['log_ac_resid'].std()
abs_floor = min_acres # acres — prevents flagging very small projects

# Primary flag: GIS >> MGT (polygon likely reflects larger management unit)
# Negative residuals = MGT is much lower than expected given GIS size
n_std = 3.0 # 3-acre SD
df_valid['ac_flag'] = (
    (df_valid['log_ac_resid'] < -n_std * resid_std) &
    (df_valid['acres_diff'] > abs_floor)
)
print(f"\nFlag summary (n_std={n_std}, abs_floor={abs_floor} acres):")
print(f"\n\tGIS-inflated (polygon too large): {df_valid['ac_flag'].sum()}")

# --- Merge flags back to original dataframe
log_cols = ['ac_flag', 'log_ac_pred', 'log_ac_resid']
cft_l = cft_.merge(df_valid[['OBJECTID'] + log_cols], on='OBJECTID', how='left')
cft_l['ac_flag'] = cft_l['ac_flag'].fillna(False).astype(bool)
cft_l[['ac_flag', 'log_ac_pred', 'gis_acres']].head()

In [ ]:
"""
Plot the distribution of filtered rows
"""

fig, axes = plt.subplots(2, 2, figsize=(7,5))

# --- Make a valid mask
mask_valid = (cft_l['gis_acres'] > 0) & (cft_l['ACRES_MGT'] > 0)
df = cft_l[mask_valid].copy()

df['log_gis'] = np.log10(df['gis_acres'])
df['log_mgt'] = np.log10(df['ACRES_MGT'])

retained = df[~df['ac_flag']]
flagged  = df[df['ac_flag']]

# --- Panel 1: log-log scatter
ax = axes[0, 0]
ax.scatter(retained['log_gis'], retained['log_mgt'],
           s=4, alpha=0.3, color='steelblue', label='Retained', rasterized=True)
ax.scatter(flagged['log_gis'], flagged['log_mgt'],
           s=12, alpha=0.7, color='darkorange', label='Flagged', zorder=3)
# OLS line
x_range = np.linspace(df['log_gis'].min(), df['log_gis'].max(), 100)
slope, intercept, *_ = stats.linregress(df['log_gis'], df['log_mgt'])
ax.plot(x_range, intercept + slope * x_range, color='gray', lw=1.2,
        ls='--', label='OLS fit')
ax.set_xlabel('log10(ACRES_GIS)'); ax.set_ylabel('log10(ACRES_MGT)')
ax.set_title('Log-log scatter', fontsize=10)
ax.legend(fontsize=8, markerscale=1.5)

# --- Panel 2: residual distribution
ax = axes[0, 1]
resid_std = df['log_ac_resid'].std()
threshold = -3 * resid_std
bins = np.linspace(df['log_ac_resid'].min(), df['log_ac_resid'].max(), 50)
ax.hist(retained['log_ac_resid'], bins=bins, color='steelblue',
        alpha=0.7, label='Retained')
ax.hist(flagged['log_ac_resid'], bins=bins, color='darkorange',
        alpha=0.8, label='Flagged')
ax.axvline(threshold, color='crimson', ls='--', lw=1.5,
           label=f'−3σ = {threshold:.2f}')
ax.set_xlabel('Residual (log10 scale)'); ax.set_ylabel('Count')
ax.set_title('Residual distribution', fontsize=10)
ax.legend(fontsize=8)

# --- Panel 3: flagged records by activity
ax = axes[1, 0]
act_stats = (flagged.groupby('ACTIVITY')['acres_diff']
             .median()
             .sort_values(ascending=True))

ax.barh(act_stats.index, act_stats.values, color='darkorange', alpha=0.8)

for i, (activity, val) in enumerate(act_stats.items()):
    n = len(flagged[flagged['ACTIVITY'] == activity])
    ax.text(val + 15, i, f'n={n}, med={val:.0f} ac',
            va='center', fontsize=7, color='dimgray')

# extend xlim to give labels room
ax.set_xlim(0, act_stats.max() * 1.5)
ax.set_xlabel('Median acres_diff (flagged records)')
ax.set_title('Flagged records by activity', fontsize=10)
ax.tick_params(axis='y', labelsize=8)

# --- Panel 4: flag rate by GIS acre size bin
ax = axes[1, 1]
bin_edges = [0, 1, 10, 100, 1000, 10000, np.inf]
bin_labels = ['<1', '1–10', '10–100', '100–1k', '1k–10k', '>10k']
df['size_bin'] = pd.cut(df['ACRES_GIS'], bins=bin_edges, labels=bin_labels)
flag_rates = (df.groupby('size_bin', observed=True)['ac_flag']
              .mean() * 100)
ax.bar(flag_rates.index, flag_rates.values, color='steelblue', alpha=0.8)
overall_rate = df['ac_flag'].mean() * 100
ax.axhline(overall_rate, color='gray', ls='--', lw=1.2,
           label=f'Overall {overall_rate:.1f}%')
ax.set_xlabel('ACRES_GIS (log-scale bin)')
ax.set_ylabel('Flag rate (%)')
ax.set_title('Flag rate by size class', fontsize=10)
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
out_png = join(projdir,'figures/LogLog_Acres_Diagnostics.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,3))

sc = ax.scatter(
    df['acres_diff'],
    df['log_ac_resid'],
    c=df['ac_flag'].astype(int),
    cmap='coolwarm', s=4, alpha=0.4, rasterized=True
)
ax.axhline(-n_std * resid_std, color='crimson', ls='--', lw=1.2,
           label=f'−3σ residual threshold')
ax.axvline(2000, color='darkorange', ls='--', lw=1.2,
           label='2,000 ac absolute threshold')
ax.set_xscale('log')
ax.set_xlabel('acres_diff (log scale)')
ax.set_ylabel('log10 residual')
ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
# --- Make another flag for absolute difference (>2,000 acres)
abs_by_activity = (cft_l.groupby('ACTIVITY')['acres_diff']
                   .transform(lambda x: x.quantile(0.99)))
cft_l['ac_flag_abs'] = cft_l['acres_diff'] > abs_by_activity
print(f"Treatments with an absolute acre difference flag: "
      f"{len(cft_l[cft_l['ac_flag_abs']])}")

In [ ]:
# --- Save this file out
out_fp = join(projdir,'data/spatial/mod/CFTv2_CO_AllTreatments_Flagged.gpkg')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
cft_l.to_file(out_fp)
print(f"Saved to {out_fp}")

## Create treatment interactions

In this section, we create the full treatments interactions table which distills overlapping treatments into a unified layer with attributes describing the effective treatment type (for TEALOM modeling), first/last treatment type and date, and full treatment set.

In [ ]:
cft_l['ACTIVITY'].unique()

In [ ]:
# --- Keep valid geometries
from tealom.treatments import make_valid_nonempty, make_geom_key

# --- Make a WKB hashable geometry key to identify exact matches
SNAP = 1 # 1-meter snap tolerance
cft_l['geometry'] = cft_l.geometry.apply(
    lambda g: shapely.set_precision(g, SNAP) if (g is not None and not g.is_empty) else g
)

cft_l = make_valid_nonempty(cft_l)

cft_l['geom_key'] = cft_l.geometry.apply(make_geom_key)
print(f"Geom Keys:\n{cft_l.geom_key.head()}")

In [ ]:
# --- Run a snap on geometries to handle slight offsets


In [ ]:
# --- Run a dissolve to handle duplicate entries
diss_cols = ['AGENCY_C', 'ACTIVITY', 'MGT_TYPE_GP', 'YEAR_COMP']
cft_twig_d = (
    cft_twig_d
    .dissolve(by=diss_cols, dropna=False, as_index=False)
    .explode(index_parts=False)
)
print(len(cft_trts_d))

# --- Add geometry key
# repair geometries first
cft_twig_d["geometry"] = cft_twig_d.geometry.make_valid()
GRID = 0.1  # precision snap tolerance in map units
cft_twig_d["geometry"] = cft_trts_d.geometry.apply(
    lambda g: shapely.set_precision(g, GRID) if (g is not None and not g.is_empty) else g
)
cft_twig_d = cft_twig_d[~cft_twig_d.geometry.is_empty].copy()
cft_twig_d["geom_key"] = cft_twig_d.geometry.apply(make_geom_key)

# --- Recalculate GIS Acres
cft_twig_d['GIS_ACRES'] = cft_twig_d.geometry.area / 4046.86 # calculate the GIS acres

# --- Subset columns
cft_twig_d = cft_twig_d[
    ['PROJ_ID','AGENCY_C','YEAR_COMP','MGT_TYPE_GP','ACTIVITY',
     'GIS_ACRES','geom_key','geometry']
]
print(cft_twig_d.head())